# Updater handler

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
from sweat.timeseries import status_handler as sh
from sweat.timeseries import updater_handler as uh
from sweat.timeseries.linear_updater import LinearUpdaterParams
from sweat.timeseries.types import TimeSeriesVar as TSVar

### Create a updater and update a time series

In [ ]:
config = {
    "method": "linear",
    "params": {"strict_mode": True, "radiation_mode": "EXTERNAL"},
}

In [ ]:
LinearUpdaterParams.model_validate(config["params"])

In [ ]:
uh.UpdaterConfig.model_validate(config)

In [ ]:
config = {
    "method": "linear",
}

In [ ]:
updater_config = uh.UpdaterConfig.model_validate(config)

In [ ]:
updater_config

In [ ]:
updater = uh.create(method=updater_config.method, params=updater_config.params)

In [ ]:
# Setup data
window_size = 7
today = dt.datetime.now(tz=dt.UTC).date()
dates = np.array(pd.date_range(end=today, periods=window_size).to_list())
ts = xr.Dataset(
    {
        TSVar.ET.value: (
            ["x", "y", "time"],
            np.array(
                [
                    [
                        [1.2, 1.4, 1.6, 1.8, 1.8, 1.8, np.nan],
                        [1.4, 1.4, 1.4, 1.4, 1.4, 1.4, np.nan],
                    ]
                ]
            ),
        ),
        TSVar.RADIATION.value: (
            ["x", "y", "time"],
            np.array(
                [
                    [
                        [200, 200, 200, 200, 200, 200, 200],
                        [200, 200, 200, 200, 200, 200, 200],
                    ]
                ]
            ),
        ),
        TSVar.FLAGS.value: (
            ["x", "y", "time"],
            np.array(
                [
                    [
                        [0, 386, 386, 0, 131, 132, 1],
                        [132, 0, 131, 259, 387, 515, 1],
                    ]
                ],
                dtype=sh.STATUS_TYPE,
            ),
        ),
    },
    coords={
        "time": np.array(
            pd.date_range(end=today, periods=window_size).to_list()
        ),
        "x": np.arange(1),
        "y": np.arange(2),
    },
)
ts = ts.transpose("time", "x", "y")
acquisition_dates = [1, 3, 5, 6]
feed_dates = dates[acquisition_dates]
feed = xr.Dataset(
    {
        TSVar.ET.value: (
            ["x", "y", "time"],
            np.array(
                [[[np.nan, 1.8, np.nan, 2.4], [1.4, np.nan, 2.2, np.nan]]]
            ),
        ),
        TSVar.VALID.value: (
            ["x", "y", "time"],
            np.array([[[0, 1, 0, 1], [1, 0, 1, 0]]]),
        ),
    },
    coords={
        "time": feed_dates,
        "x": np.arange(1),
        "y": np.arange(2),
    },
)
feed = feed.transpose("time", "x", "y")

In [ ]:
updater.update(ts, feed)

### Run 

In [ ]:
config = {
    "method": "linear",
}

In [ ]:
uh.run(ts, feed=feed, config=config)